In [1]:
import math
import time
from enum import Enum, auto
# my utils
from utils import *

import cv2
import numpy as np

from filterpy.common import Saver 

# Mac screen info
from AppKit import NSScreen
SCREEN_INDEX = 0
SCREEN_WIDTH, SCREEN_HIGHT = NSScreen.screens()[SCREEN_INDEX].frame().size.width, NSScreen.screens()[SCREEN_INDEX].frame().size.height
print(f'screen size: {(SCREEN_WIDTH, SCREEN_HIGHT)}')

# how to control your mouse: https://stackoverflow.com/questions/281133/how-to-control-the-mouse-in-mac-using-python
# mouse controll: https://pypi.org/project/pynput/
from pynput import mouse as Mouse

# hand landmark: https://google.github.io/mediapipe/solutions/hands.html
import mediapipe as mp 
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

screen size: (1920.0, 1200.0)


In [2]:
def draw_handedness(img, handedness):
  if handedness[0] < 0.5:
    label = 'Left' 
  elif handedness[0] > 0.5:
    label = 'Right' 
  else:
    label = 'Unknown'

  img_w, img_h = img.shape[1], img.shape[0]
  cv2.putText(img, f'{label} {np.abs((handedness[0]-.5)*2) :.2f}',
      org=(img_w//2 - 200, 30), # LeftBottomCornerOfText
      fontFace=cv2.FONT_HERSHEY_SIMPLEX, 
      fontScale=1,
      color=(0, 0, 255),
      lineType=2)

def draw_landmarks(img, landmarks, color=(50, 255, 50)):
  img_w, img_h = img.shape[1], img.shape[0]
  # draw dot
  for i in range(21):
    p = landmarks[i, 0:2, 0]
    cv2.circle(img, p.astype(int), radius=4, thickness=-1, color=(50, 50, 255))

  # draw line 
  pairs = []
  # wrist to index~pinky
  pairs.append((0, 1))
  pairs.append((0, 5))
  pairs.append((0, 17))
  # finger to finger
  pairs.append((5, 9))
  pairs.append((9, 13))
  pairs.append((13, 17))
  # mcp to tip
  for i in range(1, 21, 4):
    for j in range(3):
      pairs.append((i+j, i+j+1))

  for pp1, pp2 in pairs:
    p1 = landmarks[pp1, 0:2, 0]
    p2 = landmarks[pp2, 0:2, 0]
    cv2.line(img, p1.astype(int), p2.astype(int), thickness=2, color=color)

def draw_finger_state(img, handedness, finger_states):
  img_w, img_h = img.shape[1], img.shape[0]

  state_str = ''
  for finger_idx in range(5): 
    if handedness[0] < 0.5:
      state_str = f'{finger_states[finger_idx].value}{state_str}'
    else:
      state_str = f'{state_str}{finger_states[finger_idx].value}'      
  cv2.putText(img, state_str,
      org=(img_w//2, 30), # bottomLeftCornerOfText
      fontFace=cv2.FONT_HERSHEY_SIMPLEX, 
      fontScale=1,
      color=(0, 0, 255),
      lineType=2)


![21 hand landmarks](https://google.github.io/mediapipe/images/mobile/hand_landmarks.png)

## Spec

### Tracking
- existence
  - shape: (1,)
  - range: [0, 1]
- handedness
  - shape: (1,)
  - range: [0, 1] (0: left, 1: right)
- landmarks
  - shape: (21, 3,)
  - range: [0, 1] (scale with `image_width`, `image_height`, `image_width`)

### Filter
- existence
  - *state shape: (2,) (position, velocity)*
  - *z shape: (1,) (position)*
  - x: [0.5, 0].T
  - F: [[1, dt], [0, 1]]
  - H: [1, 0]
  - init_P: eye(2,2) * 0.5
  - init_Q: eye(2,2) * 0 **(not sure)**
  - init_R: 0.2
- handedness 
  - *state shape: (2,) (position, velocity)*
  - *z shape: (1,) (position)*
  - x: [0.5, 0].T
  - F: [[1, dt], [0, 1]]
  - H: [1, 0]
  - init_P: eye(2,2) * 0.5
  - init_Q: eye(2,2) * 0 **(not sure)**
  - init_R: 0.5 **(tuning)**
- landmarks 
  - *state shape: (21 * 3 * 2,) = (126,) ((landmarks) (x,y,z) (position, velocity))*
  - *z shape: (21 * 3,) = (63,) ((landmarks) (x,y,z))*
  - x: [n=126, value=0.5].T
  - F: [[1, dt, 0, ...], [0, 1, dt, 0, ...], [0, 0, 1, dt, 0, ...] ... [dt, 0, ..., 1]]
  - H: [1, 0, 1, 0, ...]
  - init_P: eye(126,126) * 0.5
  - init_Q: eye(126,126) * 0 **(not sure)**
  - init_R: 0.01 **(should be low)**

In [3]:
DESIRED_HEIGHT = 720
DESIRED_WIDTH = 720
def preprocess_img(image):
  h, w = image.shape[:2]
  # resize
  if h < w:
    img = cv2.resize(image, (DESIRED_WIDTH, math.floor(h/(w/DESIRED_WIDTH))))
  else:
    img = cv2.resize(image, (math.floor(w/(h/DESIRED_HEIGHT)), DESIRED_HEIGHT))

  return img

def landmarks2vec(hand_landmarks):
  vecs = np.empty((21, 3))
  for landmark_idx in mp_hands.HandLandmark:
    vecs[landmark_idx] = np.array([
      hand_landmarks.landmark[landmark_idx].x,
      hand_landmarks.landmark[landmark_idx].y,
      hand_landmarks.landmark[landmark_idx].z,
    ])
  return vecs

In [4]:
def switch_hand(landmarks):
  landmarks = np.array(landmarks)
  landmarks[[1, 2, 3, 4, 5, 6, 7, 8]] = landmarks[[17, 18, 19, 20, 13, 14, 15, 16]] 
  return landmarks 

In [5]:
def shift_to_palm_coordinate(landmarks, is_right_hand):
  """
  origin: wrist
  y-axis: index-mcp to wrist vector
  z-axis: orthogonal to y-axis vector and wrist pinky-mcp vector, point out of palm 
  @return shifted_landmarks, rotation_matrix
  """
  landmarks = np.array(landmarks)

  # shift origin
  delta_origin = np.copy(landmarks[0])
  landmarks -= delta_origin 

  # calculate rotation martix
  axis_y = landmarks[5] - landmarks[0]
  axis_z = np.cross(landmarks[9] - landmarks[0], axis_y)
  # correct z-axis direction
  axis_z = axis_z if is_right_hand else -axis_z
  r = get_coordinate_rotation_matrix([-axis_y, -axis_z], [[0,1,0], [0,0,1]])
  
  return landmarks @ r, r,delta_origin 


def shift_back_to_origin_coordinate(landmarks, rotation_matrix, delta_origin):
  landmarks = np.array(landmarks)

  # rotate back
  # NOTE: invsere of rotation martix = transpose of rotation matrix
  landmarks = landmarks @ rotation_matrix.T
  
  # shift origin
  landmarks += delta_origin
  
  return landmarks


def annotate_3axis(img, rotation_matrix, delta_origin):
  start = shift_back_to_origin_coordinate([0,0,0], rotation_matrix, delta_origin)
  x_end = shift_back_to_origin_coordinate([40,0,0], rotation_matrix, delta_origin)
  y_end = shift_back_to_origin_coordinate([0,40,0], rotation_matrix, delta_origin)
  z_end = shift_back_to_origin_coordinate([0,0,40], rotation_matrix, delta_origin)
  cv2.arrowedLine(img, (start[0:2]).astype(int), (x_end[0:2]).astype(int), thickness=2, color=(0, 0, 255))
  cv2.arrowedLine(img, (start[0:2]).astype(int), (y_end[0:2]).astype(int), thickness=2, color=(0, 255, 0))
  cv2.arrowedLine(img, (start[0:2]).astype(int), (z_end[0:2]).astype(int), thickness=2, color=(255, 0, 0))
  cv2.putText(img, 'x', org=((x_end[0:2] + [5,5]).astype(int)), color=(0, 0, 255), fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=.7, lineType=2)
  cv2.putText(img, 'y', org=((y_end[0:2] + [5,5]).astype(int)), color=(0, 255, 0), fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=.7, lineType=2)
  cv2.putText(img, 'z', org=((z_end[0:2] + [5,5]).astype(int)), color=(255, 0, 0), fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=.7, lineType=2)

In [6]:
class FINGER_STATE(Enum):
  BENT = 0
  STRAIGHT = 1
  UNKNOWN = auto()

class FINGER(Enum):
  THUMB = 0
  INDEX = 1
  MIDDLE = 2
  RING = 3
  PINKY = 4


def get_finger_state(landmarks):
  res = np.array([FINGER_STATE.UNKNOWN for i in range(5)])
  
  # finger_idx
  # thumb: 1~4, index: 5~8, middle: 9~12, ring: 13~16, pinky: 17~20
  for finger_idx in range(1, 21, 4):
    tip = landmarks[finger_idx+3, :, 0]
    dip = landmarks[finger_idx+2, :, 0]
    pip = landmarks[finger_idx+1, :, 0]
    mcp = landmarks[finger_idx, :, 0]

    # print(f'{finger_name[(finger_idx-1)//4]} finger')
    # print(f'{np.pi - angle_between_vectors(pip - mcp, pip - dip)}', end='')
    # print(f'{np.pi - angle_between_vectors(dip - pip, dip - tip)}')
    

    accumulated_angle = (np.pi - angle_between_vectors(pip - mcp, pip - dip)) + (np.pi - angle_between_vectors(dip - pip, dip - tip))
    
    if accumulated_angle > (np.pi * 0.4):
      res[(finger_idx-1) // 4] = FINGER_STATE.BENT
    else:
      res[(finger_idx-1) // 4] = FINGER_STATE.STRAIGHT

  return res

In [9]:
mouse = Mouse.Controller()
# mouse_listener = Mouse.Listener(
#   on_move=lambda x,y: print(f'Pointer move to ({x :.2f}, {y :.2f})'),
#   on_scroll=lambda x,y,dx,dy: print(f'Scrolled ({x :.2f}, {y :.2f}) at {"down" if dy < 0 else "up"}'))
# mouse_listener.start()
# mouse_listener.wait()

# fps
s_time = time.time()
frame_cnt = 0
prev_frame_cnt = 0
prev_timestamp = time.time()

# init estimate
existence = [.5, 0]
handedness = [.5, 0]
landmarks = np.full((21, 3, 2), 0.5)

# filter
dt = 1./10
existence_f = pos_vel_filter(existence, P=.5, R=.5, Q=.001, dt=1.)
handedness_f = pos_vel_filter(handedness, P=.5, R=.1, Q=.01, dt=1.)
x_Q = np.eye(2,2) * [15., 200.]
y_Q = np.eye(2,2) * [25., 200.]
z_Q = np.eye(2,2) * [25., 50.]
ls_Q = np.block([[x_Q, np.zeros((2,2)), np.zeros((2,2))], [np.zeros((2,2)), y_Q, np.zeros((2,2))], [np.zeros((2,2)), np.zeros((2,2)), z_Q]])
x_R = 1.
y_R = 1.
z_R = 5.
ls_R = np.eye(3,3) * [x_R, y_R, z_R]
landmarks_f = multi_pos_vel_filter(landmarks.flatten(), P=DESIRED_WIDTH/3., R=block_diagonal_array(63//3, ls_R), Q=block_diagonal_array(126//6, ls_Q), dt=1./15)
landmarks_s = Saver(landmarks_f)

# gesture

# DEBUG: params
tmax = -100


cap = cv2.VideoCapture(0)
with mp_hands.Hands(
    min_detection_confidence=0.75,
    min_tracking_confidence=0.7) as hands:
  while cap.isOpened():
    success, raw_image = cap.read()
    if not success:
      print("Ignoring empty camera frame.")
      # If loading a video, use 'break' instead of 'continue'.
      continue

    # Flip the image horizontally for a later selfie-view display, and convert the BGR image to RGB.
    image = cv2.cvtColor(cv2.flip(preprocess_img(raw_image), 1), cv2.COLOR_BGR2RGB)
    # To improve performance, optionally mark the image as not writeable to pass by reference.
    image.flags.writeable = False
    results = hands.process(image)
    
    image_hight, image_width, _ = image.shape
    # Draw the hand annotations on the image.
    image.flags.writeable = True
    annotated_image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)


    # existence detection
    if results.multi_hand_landmarks:
      z_existence = 1
    else :
      z_existence = 0
    existence_f.predict()
    existence_f.update(z_existence)
    existence = existence_f.x
      

    if existence[0] > 0.5:

      handedness_f.predict()
      landmarks_f.predict()

      if z_existence == 1:
        # NOTE: `existence` only response for one hand existence
        raw_handedness, raw_landmarks = results.multi_handedness[0], results.multi_hand_landmarks[0]

        # update handedness 
        z_handedness = .5 + raw_handedness.classification[0].score * (.5 if raw_handedness.classification[0].index == 1 else -.5)
        handedness_f.update(z_handedness)

        z_landmarks = landmarks2vec(raw_landmarks)
        # switch hand landmarks
        should_switch_hand = (handedness[0] > 0.5 and z_handedness < 0.5) or (handedness[0] < 0.5 and z_handedness > 0.5)
        if should_switch_hand: 
          z_landmarks = switch_hand(z_landmarks)
        # scale to image size
        z_landmarks *= [image_width, image_hight, image_width] 
        # change coordinate
        # z_shift_landmarks, rotation_matrix, delta_origin = shift_to_palm_coordinate(z_landmarks, handedness[0] > 0.5) 
        # update landmarks
        landmarks_f.update(z_landmarks.flatten())
        # landmarks_f.update(z_shift_landmarks.flatten())
        
        # save landmarks
        # landmarks_s.save()

      handedness = handedness_f.x
      landmarks = landmarks_f.x.reshape(21, 3, 2)


      # Gesture
      finger_states = get_finger_state(landmarks)

      ## move mouse
      if np.all(finger_states[[2,3]] == FINGER_STATE.BENT) and (
            # (np.all(np.abs(landmarks[0, 0:2, 1]) < 10.))
            # or 
            (finger_states[1] == FINGER_STATE.STRAIGHT)
          ):

        max_move_speed = 100
        x = np.clip(landmarks[8, 0, 1] , -max_move_speed, max_move_speed)
        y = np.clip(landmarks[8, 1, 1] , -max_move_speed, max_move_speed)

        # WARN: mouse may move to the negative position, which refer to second monitor 
        # clip mouse position in the "main monitor"
        # FUTURE: support multi-monitor
        move_x = np.clip(x, 0 - mouse.position[0], SCREEN_WIDTH - mouse.position[0])
        move_y = np.clip(y, 0 - mouse.position[1], SCREEN_HIGHT - mouse.position[1])
        mouse.move(move_x, move_y)

      # ## scroll vertical
      # # TODO: smooth scroll (keep scroll after gesture disappear)

      # ## scroll up 
      max_scroll_speed = 5
      scale_factor = .07
      if np.all(finger_states[[4]] == FINGER_STATE.BENT) and \
            np.all(finger_states[[1,2,3]] == FINGER_STATE.STRAIGHT):
        scroll_y = np.clip(-landmarks[8, 1, 1]*scale_factor, 0, max_scroll_speed)
        mouse.scroll(0, scroll_y)
        # annotate
        cv2.arrowedLine(annotated_image, landmarks[8, 0:2, 0].astype(int), (landmarks[8, 0:2, 0] + [0, -scroll_y*10]).astype(int), color=(255, 50, 50), thickness=3)

      # ## scroll down
      if np.all(finger_states[[3,4]] == FINGER_STATE.BENT) and \
            np.all(finger_states[[1,2]] == FINGER_STATE.STRAIGHT):
        scroll_y = np.clip(landmarks[8, 1, 1]*scale_factor, -max_scroll_speed, 0)
        mouse.scroll(0, scroll_y)
        # annotate
        cv2.arrowedLine(annotated_image, landmarks[8, 0:2, 0].astype(int), (landmarks[8, 0:2, 0] + [0, -scroll_y*10]).astype(int), color=(255, 50, 50), thickness=3)

      ## switch desktop
      ## switch screen


      # Draw 
      # measurement landmarks
      # mp_drawing.draw_landmarks(annotated_image, raw_landmarks, mp_hands.HAND_CONNECTIONS)
      # draw_landmarks(annotated_image, np.stack([z_landmarks, np.zeros(z_landmarks.shape)], axis=2), (255, 50, 50, 0.5))

      # Draw normal landmark
      draw_handedness(annotated_image, handedness)

      # Draw projected landmarks
      # inv_landmarks = np.stack([
        # shift_back_to_origin_coordinate(landmarks[:, :, 0], rotation_matrix, delta_origin), 
        # shift_back_to_origin_coordinate(landmarks[:, :, 1], rotation_matrix, delta_origin)], axis=2)
      # Draw projection axis
      # annotate_3axis(annotated_image, rotation_matrix, delta_origin)      

      # Draw handedness
      draw_landmarks(annotated_image, landmarks)

      # Drw finger states
      draw_finger_state(annotated_image, handedness, finger_states)
      

      # DEBUG: output 
      # if time.time() - s_time > 1:
        # tmax = np.max([tmax, np.abs(landmarks_s.z[-1].reshape(21,3)[8, 2] - landmarks_s.z[-2].reshape(21,3)[8, 2]) / dt])
      # tmax = np.max([tmax, np.abs(landmarks_f.x.reshape(21,3,2)[8,2,1])])
      # cv2.putText(annotated_image, 
          #  f'{landmarks_f.x.reshape(21,3,2)[8,1,1] :.2f}',
          #  f'{scroll_y :.2f}',
          #  org=(int(image_width*.01), int(image_hight*.15)), # bottomLeftCornerOfText
          #  fontFace=cv2.FONT_HERSHEY_SIMPLEX, 
          #  fontScale=.8,
          #  color=(255, 255, 255),
          #  lineType=2)

    else: 
      pass
      # print('-------------- no hand found --------------')


    # Draw fps
    frame_cnt += 1
    now_timestamp = time.time()
    if now_timestamp - prev_timestamp >= 1:
      prev_timestamp, prev_frame_cnt = now_timestamp, frame_cnt
      frame_cnt = 0
    cv2.putText(annotated_image, f'{prev_frame_cnt}', org=(image_width - 30, 20), fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.5, color=(100, 255, 100), lineType=2)


    cv2.imshow('Hands', annotated_image)
    if cv2.waitKey(20) & 0xFF == 27:
      # FIXME: `Listener.stop()` failed to stop mouse listerner
      # mouse_listener.stop()
      break

cap.release()
# cv2.destroyAllWindows()